<a href="https://colab.research.google.com/github/mathiyarasi05/Cardio/blob/main/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 1: Install and Upgrade Kaggle API
!pip install --upgrade -q kaggle

# STEP 2: Upload your kaggle.json file
# (Upload manually or ensure it's in Drive)

# STEP 3: Move kaggle.json to the correct location
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/ECG/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# STEP 4: Download ECG Heartbeat Dataset
!kaggle datasets download -d shayanfazeli/heartbeat

# STEP 5: Create destination folder, move zip, and unzip
!mkdir -p /content/drive/MyDrive/ECG/ECG_DATA
!mv /content/heartbeat.zip /content/drive/MyDrive/ECG/
!unzip -q /content/drive/MyDrive/ECG/heartbeat.zip -d /content/drive/MyDrive/ECG/ECG_DATA


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 7.4 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 5, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 4, in <module>
    from kaggle.api.kaggle_api_extended import KaggleApi
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 55, in <module>
    from kagglesdk import get_access_token_from_env, KaggleClient, KaggleCredentials, KaggleEnv, KaggleOAuth  # type: ignore[attr-defined]
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: cannot import name 'get_access_token_from_env' from 'kagglesdk' (unknown location)
mv: cannot stat '/content/heartbeat.zip': No such file or directory
replace /content/drive/MyDrive/ECG/ECG_DATA/mitbih_test.csv?

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [ ]:
base = "/content/drive/MyDrive/ECG/ECG_DATA"


In [ ]:
mitbih_train = pd.read_csv(base + "/mitbih_train.csv", header=None)
mitbih_test  = pd.read_csv(base + "/mitbih_test.csv", header=None)


In [ ]:
ptb_normal   = pd.read_csv(base + "/ptbdb_normal.csv", header=None)
ptb_abnormal = pd.read_csv(base + "/ptbdb_abnormal.csv", header=None)


In [ ]:
X_train_mit = mitbih_train.iloc[:, :-1].values
y_train_mit = mitbih_train.iloc[:, -1].values

X_test_mit = mitbih_test.iloc[:, :-1].values
y_test_mit = mitbih_test.iloc[:, -1].values


In [ ]:
scaler = StandardScaler()

X_train_mit = scaler.fit_transform(X_train_mit)
X_test_mit = scaler.transform(X_test_mit)


In [ ]:
X_train_mit = X_train_mit.reshape(X_train_mit.shape[0], X_train_mit.shape[1], 1)
X_test_mit  = X_test_mit.reshape(X_test_mit.shape[0], X_test_mit.shape[1], 1)

print(X_train_mit.shape, X_test_mit.shape)


(87554, 187, 1) (21892, 187, 1)


In [ ]:
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization

model = Sequential([
    Input(shape=(X_train_mit.shape[1], 1)),   # FIXED

    Conv1D(32, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(64, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(5, activation="softmax")  # MIT-BIH (5 classes)
])


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization

model = Sequential([
    Input(shape=(X_train_mit.shape[1], 1)),

    Conv1D(32, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(64, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(5, activation="softmax")
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_mit, y_train_mit,
    validation_data=(X_test_mit, y_test_mit),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 84s 59ms/step - accuracy: 0.9235 - loss: 0.2887 - val_accuracy: 0.9708 - val_loss: 0.1049
Epoch 2/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 79s 58ms/step - accuracy: 0.9720 - loss: 0.1076 - val_accuracy: 0.9753 - val_loss: 0.0938
Epoch 3/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 81s 59ms/step - accuracy: 0.9767 - loss: 0.0849 - val_accuracy: 0.9770 - val_loss: 0.0782
Epoch 4/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 81s 59ms/step - accuracy: 0.9804 - loss: 0.0688 - val_accuracy: 0.9805 - val_loss: 0.0691
Epoch 5/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 81s 59ms/step - accuracy: 0.9827 - loss: 0.0584 - val_accuracy: 0.9816 - val_loss: 0.0745
Epoch 6/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 82s 59ms/step - accuracy: 0.9837 - loss: 0.0537 - val_accuracy: 0.9840 - val_loss: 0.0662
Epoch 7/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 80s 58ms/step - accuracy: 0.9860 - loss: 0.0477 - val_accuracy: 0.9831 - val_loss: 0.0675
Epoch 8/10
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 78s 57ms/step - accuracy: 0.9876 -

In [ ]:
loss, acc = model.evaluate(X_test_mit, y_test_mit)
print("Test Accuracy:", acc)



In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Predict test data
y_pred = model.predict(X_test_mit)

# Convert probabilities to class numbers
y_pred_classes = np.argmax(y_pred, axis=1)

# Print full performance report
print(classification_report(y_test_mit, y_pred_classes))

NameError: name 'model' is not defined